# MIDAS Regression

This notebook demonstrates **Mixed Data Sampling (MIDAS)** regression for nowcasting
with mixed-frequency data.

**Reference**: Ghysels, E., Santa-Clara, P., & Valkanov, R. (2004). "The MIDAS Touch:
Mixed Data Sampling Regression Models." CIRANO Working Papers.

MIDAS directly regresses a low-frequency variable (quarterly GDP) on high-frequency
regressors (monthly indicators) using **parameterized weight functions** that avoid
the parameter proliferation problem of unrestricted distributed lag models.

In [ ]:
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from forecastbox.nowcasting import MIDAS

# Add helpers path
sys.path.insert(0, "../utils")
from helpers import load_mixed_freq, load_gdp_vintages, load_macro_brazil, get_vintage

warnings.filterwarnings("ignore")
np.random.seed(42)

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

## 1. The Mixed Frequency Problem

A natural approach to mixed-frequency data is to **aggregate** the high-frequency data
to the lower frequency (e.g., take quarterly averages of monthly data). But this throws
away information!

Consider: the pattern of monthly values within a quarter matters. A quarter where
production grows steadily is very different from one with a collapse and recovery,
even if the quarterly averages are similar.

MIDAS preserves this within-quarter information by working directly with the
high-frequency data.

In [ ]:
# Load mixed-frequency data
data = load_mixed_freq()
print(f"Dataset: {data.shape}")
print(f"Date range: {data.index[0]} to {data.index[-1]}")

# Demonstrate information loss with aggregation
ip_monthly = data["industrial_production"].dropna()
ip_quarterly_mean = ip_monthly.resample("QS").mean()

# Find quarters with similar means but different within-quarter patterns
print("\n--- Information Loss from Aggregation ---")
print("\nMonthly industrial production (showing 2 example quarters):")

# Show two quarters side-by-side
q1_data = data.loc["2017-01":"2017-03", "industrial_production"]
q2_data = data.loc["2019-01":"2019-03", "industrial_production"]
print(f"\n2017-Q1 monthly: {q1_data.values.round(2)} → Q avg: {q1_data.mean():.2f}")
print(f"2019-Q1 monthly: {q2_data.values.round(2)} → Q avg: {q2_data.mean():.2f}")
print(f"Difference in Q avg: {abs(q1_data.mean() - q2_data.mean()):.2f}")
print("\nAggregation hides the within-quarter dynamics!")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: monthly vs quarterly
axes[0].plot(ip_monthly.index, ip_monthly.values, "b-", linewidth=1, alpha=0.7, label="Monthly")
axes[0].step(ip_quarterly_mean.index, ip_quarterly_mean.values, "r-", linewidth=2.5,
             where="mid", label="Quarterly Mean")
axes[0].set_title("Monthly vs Quarterly Aggregation", fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right: within-quarter variation that is lost
ip_quarterly_std = ip_monthly.resample("QS").std()
axes[1].bar(ip_quarterly_std.index, ip_quarterly_std.values, width=60,
            color="coral", alpha=0.7, edgecolor="black")
axes[1].set_title("Within-Quarter Std Dev (Lost Information)", fontsize=12)
axes[1].set_ylabel("Std Dev")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. MIDAS with Exponential Almon Lag

The MIDAS regression model is:

$$y_t^{(Q)} = \alpha + \beta \sum_{k=0}^{K-1} w(k; \theta) x_{t-k}^{(M)} + \varepsilon_t$$

Where $w(k; \theta)$ is a parameterized weight function. The **Exponential Almon**
polynomial uses:

$$w(k; \theta) = \frac{\exp(\theta_1 k + \theta_2 k^2 + \ldots)}{\sum_j \exp(\theta_1 j + \theta_2 j^2 + \ldots)}$$

This flexible form can produce decaying, hump-shaped, or U-shaped weight patterns
with just 2-3 parameters, avoiding the curse of dimensionality.

In [ ]:
# MIDAS with Almon polynomial weights
midas_almon = MIDAS(
    target="gdp_growth",
    high_freq=["industrial_production"],
    weight_scheme="almon",
    n_lags=12,
    poly_order=2,
    freq_ratio=3,
)

midas_almon.fit(data)
print(midas_almon.summary())

# Nowcast
fc_almon = midas_almon.nowcast()
print(f"\nNowcast (Almon): {fc_almon.point[0]:.4f}")
print(f"95% CI: [{fc_almon.lower_95[0]:.4f}, {fc_almon.upper_95[0]:.4f}]")

# Plot the estimated weight function
fig, ax = plt.subplots(figsize=(10, 5))
midas_almon.plot_weights(ax=ax)
ax.set_title("Exponential Almon Lag Weights", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 3. U-MIDAS (Unrestricted)

When the number of high-frequency lags is small relative to the sample size,
we can use **U-MIDAS** (Unrestricted MIDAS) which estimates each lag weight freely
via OLS.

U-MIDAS (Foroni, Marcellino & Schumacher, 2015) is equivalent to a standard
distributed lag model — no weight parametrization is imposed.

This is set via `weight_scheme='step'` in forecastbox.

In [ ]:
# U-MIDAS (unrestricted weights via OLS)
midas_step = MIDAS(
    target="gdp_growth",
    high_freq=["industrial_production"],
    weight_scheme="step",
    n_lags=12,
    freq_ratio=3,
)

midas_step.fit(data)
print(midas_step.summary())

# Nowcast
fc_step = midas_step.nowcast()
print(f"\nNowcast (U-MIDAS): {fc_step.point[0]:.4f}")
print(f"95% CI: [{fc_step.lower_95[0]:.4f}, {fc_step.upper_95[0]:.4f}]")

# Compare weight patterns
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

midas_almon.plot_weights(ax=axes[0])
axes[0].set_title("Almon (Parametric)", fontsize=12)

midas_step.plot_weights(ax=axes[1])
axes[1].set_title("U-MIDAS (Unrestricted/Step)", fontsize=12)

plt.suptitle("Parametric vs Unrestricted Weight Functions", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 4. Comparing Lag Polynomials

MIDAS supports several weight function specifications:

| Scheme | Parameters | Shape | Best For |
|--------|-----------|-------|----------|
| **Beta** | $\theta_1, \theta_2$ | Flexible (decaying, hump, U) | General purpose |
| **Almon** | $\theta_1, \ldots, \theta_p$ | Polynomial | Smooth weight patterns |
| **Step** (U-MIDAS) | $K$ weights | Unrestricted | Small $K$, large $T$ |

The **Beta** polynomial (Ghysels et al., 2006) uses the Beta density function:
$$w(k; \theta_1, \theta_2) \propto k^{\theta_1 - 1}(1-k)^{\theta_2 - 1}$$

In [ ]:
# Compare all three weight schemes
schemes = {
    "beta": {"weight_scheme": "beta", "n_lags": 12, "poly_order": 2},
    "almon": {"weight_scheme": "almon", "n_lags": 12, "poly_order": 2},
    "step": {"weight_scheme": "step", "n_lags": 12, "poly_order": 2},
}

results_table = []
fitted_models = {}

for name, params in schemes.items():
    midas_model = MIDAS(
        target="gdp_growth",
        high_freq=["industrial_production"],
        freq_ratio=3,
        **params,
    )
    midas_model.fit(data)
    fc = midas_model.nowcast()
    fitted_models[name] = midas_model

    results_table.append({
        "scheme": name,
        "nowcast": fc.point[0],
        "residual_std": np.sqrt(midas_model._sigma2),
        "weights_sum": midas_model.weights_.sum(),
        "max_weight_lag": int(np.argmax(midas_model.weights_)),
    })

results_df = pd.DataFrame(results_table)
print("Comparison of MIDAS Weight Schemes:")
print(results_df.to_string(index=False))

# Plot all weight functions together
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
colors = {"beta": "steelblue", "almon": "darkorange", "step": "forestgreen"}

for ax, (name, model) in zip(axes, fitted_models.items()):
    lags = np.arange(model.n_lags)
    ax.bar(lags, model.weights_, color=colors[name], alpha=0.7, edgecolor="black")
    ax.set_title(f"{name.capitalize()} Weights", fontsize=12)
    ax.set_xlabel("Lag")
    ax.set_ylabel("Weight")
    ax.grid(True, alpha=0.3, axis="y")

plt.suptitle("MIDAS Weight Functions: Beta vs Almon vs Step", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Overlay all weight functions
fig, ax = plt.subplots(figsize=(10, 5))
for name, model in fitted_models.items():
    lags = np.arange(model.n_lags)
    ax.plot(lags, model.weights_, "o-", color=colors[name], linewidth=2,
            markersize=6, label=name.capitalize())
ax.set_xlabel("Lag (months)")
ax.set_ylabel("Weight")
ax.set_title("Weight Function Comparison", fontsize=13, fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. MIDAS with GDP Vintages

In practice, GDP data is **revised** multiple times after the initial release.
The first estimate ("flash" or "advance") may differ significantly from the final value.

To evaluate nowcast accuracy in a realistic setting, we use the `gdp_vintages.csv`
dataset which contains GDP growth at different revision stages.

We compare MIDAS nowcasts against each vintage to understand how well the model
predicts both the initial release and the final revised figure.

In [ ]:
# Load GDP vintages
vintages_df = load_gdp_vintages()
print(f"GDP Vintages dataset: {vintages_df.shape}")
print(f"Vintages available: {sorted(vintages_df['vintage'].unique())}")
print(f"\nFirst few rows:")
print(vintages_df.head(10).to_string(index=False))

# Extract specific vintages
vintage_1 = get_vintage(vintages_df, 1)  # First release
vintage_5 = get_vintage(vintages_df, 5)  # Final revised

print(f"\nRevision statistics (vintage 1 → vintage 5):")
revisions = vintage_5["gdp_growth"] - vintage_1["gdp_growth"]
print(f"  Mean absolute revision: {revisions.abs().mean():.4f}")
print(f"  Max revision: {revisions.abs().max():.4f}")
print(f"  Revision std: {revisions.std():.4f}")

# Run MIDAS nowcast and compare against each vintage
print("\n--- MIDAS Nowcast vs GDP Vintages ---")

# Fit MIDAS on the mixed_freq data
midas_eval = MIDAS(
    target="gdp_growth",
    high_freq=["industrial_production"],
    weight_scheme="beta",
    n_lags=12,
    freq_ratio=3,
)
midas_eval.fit(data)

# For each quarter, compare nowcast to different vintages
eval_quarters = vintage_1.index[-8:]  # Last 8 quarters
gdp_dates = data["gdp_growth"].dropna().index

vintage_comparison = []

for q_date in eval_quarters:
    # Find closest GDP date
    closest = gdp_dates[gdp_dates <= q_date]
    if len(closest) == 0:
        continue

    for v in [1, 3, 5]:
        vdata = get_vintage(vintages_df, v)
        if q_date in vdata.index:
            actual = vdata.loc[q_date, "gdp_growth"]
            # Get MIDAS nowcast for this quarter
            fc = midas_eval.nowcast()
            vintage_comparison.append({
                "quarter": q_date,
                "vintage": v,
                "actual_gdp": actual,
                "nowcast": fc.point[0],
                "error": fc.point[0] - actual,
            })

comp_df = pd.DataFrame(vintage_comparison)
if len(comp_df) > 0:
    print("\nRMSE by vintage:")
    for v in comp_df["vintage"].unique():
        subset = comp_df[comp_df["vintage"] == v]
        rmse = np.sqrt(np.mean(subset["error"] ** 2))
        print(f"  Vintage {v}: RMSE = {rmse:.4f}")

# Visualize GDP revisions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: GDP at different vintages
for v in [1, 3, 5]:
    vdata = get_vintage(vintages_df, v)
    axes[0].plot(vdata.index, vdata["gdp_growth"], "o-", linewidth=1.5,
                 markersize=4, label=f"Vintage {v}", alpha=0.8)
axes[0].set_title("GDP Growth Across Vintages", fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_ylabel("GDP Growth")

# Right: revision distribution
axes[1].hist(revisions.values, bins=15, color="coral", alpha=0.7, edgecolor="black")
axes[1].axvline(0, color="black", linewidth=1.5)
axes[1].set_title("Distribution of GDP Revisions (V1 → V5)", fontsize=12)
axes[1].set_xlabel("Revision")
axes[1].set_ylabel("Frequency")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Exercise 1: MIDAS with different frequencies (daily financial data)

Generate synthetic daily financial data (e.g., stock returns, exchange rate changes)
and use MIDAS with `freq_ratio=22` (approx. trading days per month) to nowcast
monthly GDP. Compare with `freq_ratio=3` (quarterly/monthly).

In [ ]:
# TODO: Exercise 1
# Hints:
# 1. Generate daily data: pd.date_range('2015-01-01', '2024-08-01', freq='B')
# 2. Create synthetic daily returns: np.random.normal(0, 0.01, n_days)
# 3. Merge with mixed_freq GDP data
# 4. MIDAS(target='gdp_growth', high_freq=['daily_returns'],
#          weight_scheme='beta', n_lags=66, freq_ratio=22)
# 5. Compare weight patterns with the monthly MIDAS

### Exercise 2: Compare MIDAS, bridge and DFM for GDP nowcasting

Run a comprehensive comparison of all three nowcasting approaches using the
`mixed_freq.csv` and `macro_brazil.csv` datasets. Which method works best
in different scenarios?

In [ ]:
# TODO: Exercise 2
# Hints:
# 1. Import DFMNowcaster and BridgeEquation from forecastbox.nowcasting
# 2. For each dataset, fit all three models
# 3. Run a pseudo real-time exercise (see Notebook 01)
# 4. Compute RMSE, MAE, and bias for each model
# 5. Create a summary table and discussion
# 6. Consider: DFM is best with many indicators,
#    Bridge is simplest and most robust,
#    MIDAS preserves within-period dynamics